# EE 413 — Member 3 Final Part  
## Wavelet Compression Analysis for ResNet-18 and MobileNetV2

Student: Abdulrahman Alburaikan  
Role: Member 3  
Project: Image Compression and Classification with Deep Learning

Use this notebook **after Member 1 and Member 2 finish**.

This notebook completes the required Member 3 part:
1. Apply wavelet compression to validation images.
2. Use compression ratios **2:1, 5:1, and 10:1**.
3. Evaluate **ResNet-18** from Member 1.
4. Evaluate **MobileNetV2** from Member 2.
5. Compare both models under compression.
6. Generate figures, CSV files, JSON files, and a ZIP package for GitHub.

## Cell 27 — Member 3 imports and paths

This starts after:
- Member 1 cells 1–25
- Member 2 cells 1–26

It does not train a new model. It only evaluates trained checkpoints under wavelet compression.

In [ ]:
# Cell 27: Member 3 imports and paths

import os
import json
import random
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, datasets, models
from torchvision.models import ResNet18_Weights, MobileNet_V2_Weights

try:
    import pywt
except ImportError:
    !pip install -q PyWavelets
    import pywt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Reproducibility
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Same settings as Member 1 and Member 2
IMG_SIZE = 96
IMAGE_SIZE = 96
BATCH_SIZE = 64
NUM_WORKERS = 2

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

COMPRESSION_RATIOS = [2, 5, 10]
WAVELET_NAME = "haar"
WAVELET_LEVEL = 2
THRESHOLD_MODE = "hard"

CONTENT = Path("/content")

# Common output folders from Member 1 and Member 2 notebooks
M1_OUTPUT = CONTENT / "outputs"
M2_OUTPUT = CONTENT / "outputs_m2"

# Checkpoint paths
RESNET18_CKPT = M1_OUTPUT / "resnet18_best.pth"
MOBILENETV2_CKPT = M2_OUTPUT / "mobilenetv2_best.pth"

# Fallback if files are uploaded directly to /content
if not RESNET18_CKPT.exists():
    RESNET18_CKPT = CONTENT / "resnet18_best.pth"

if not MOBILENETV2_CKPT.exists():
    MOBILENETV2_CKPT = CONTENT / "mobilenetv2_best.pth"

# Baseline JSON paths
M1_BASELINE_JSON = M1_OUTPUT / "baseline_results.json"
M2_BASELINE_JSON = M2_OUTPUT / "baseline_results.json"

if not M1_BASELINE_JSON.exists():
    M1_BASELINE_JSON = CONTENT / "baseline_results.json"

# If both baseline files have the same name in /content, keep Member 2 from outputs_m2 if available.
if not M2_BASELINE_JSON.exists():
    M2_BASELINE_JSON = CONTENT / "baseline_results_m2.json"

# Member 3 output folder
M3_OUTPUT = CONTENT / "outputs_m3_wavelet"
M3_FIGURES = M3_OUTPUT / "figures"
M3_OUTPUT.mkdir(parents=True, exist_ok=True)
M3_FIGURES.mkdir(parents=True, exist_ok=True)

print("ResNet-18 checkpoint:", RESNET18_CKPT)
print("MobileNetV2 checkpoint:", MOBILENETV2_CKPT)
print("Member 3 output:", M3_OUTPUT)

## Cell 28 — Load baseline numbers

These are used only for checking and reporting:
- ResNet-18 baseline validation accuracy: from Member 1.
- MobileNetV2 baseline validation accuracy: from Member 2.

In [ ]:
# Cell 28: Load baseline numbers

def load_json_if_exists(path):
    path = Path(path)
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return {}

m1_baseline = load_json_if_exists(M1_BASELINE_JSON)
m2_baseline = load_json_if_exists(M2_BASELINE_JSON)

# Fallback known values from team results
M1_REPORTED_VAL_ACC = m1_baseline.get("best_val_accuracy", 0.764844)
M2_REPORTED_VAL_ACC = m2_baseline.get("best_val_accuracy", m2_baseline.get("val_accuracy", 0.750521))

print("Member 1 ResNet-18 reported validation accuracy:", f"{M1_REPORTED_VAL_ACC*100:.2f}%")
print("Member 2 MobileNetV2 reported validation accuracy:", f"{M2_REPORTED_VAL_ACC*100:.2f}%")

## Cell 29 — Build raw validation dataset

Wavelet compression must be applied **before ImageNet normalization**.

This cell rebuilds a raw `[0,1]` validation dataset from the same internal validation subset used by Member 1 and Member 2.

In [ ]:
# Cell 29: Build raw validation dataset

raw_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

normalize_transform = transforms.Normalize(MEAN, STD)


def unnormalize_tensor(x, mean=MEAN, std=STD):
    '''
    Convert an ImageNet-normalized tensor back to [0,1].
    Used only as a fallback if val_dataset already returns normalized images.
    '''
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return torch.clamp(x.cpu() * std + mean, 0, 1)


class RawSubsetFromImageFolder(Dataset):
    '''
    Rebuild raw images from ImageFolder or Subset(ImageFolder).
    This keeps the same validation indices but removes normalization.
    '''
    def __init__(self, dataset_or_subset, transform):
        self.transform = transform

        if isinstance(dataset_or_subset, Subset):
            self.base = dataset_or_subset.dataset
            self.indices = list(dataset_or_subset.indices)
        else:
            self.base = dataset_or_subset
            self.indices = list(range(len(dataset_or_subset)))

        if not hasattr(self.base, "samples"):
            raise ValueError("Base dataset does not have .samples. Use fallback wrapper.")

        self.classes = getattr(self.base, "classes", None)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image_path, label = self.base.samples[real_idx]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, int(label)


class RawFallbackDataset(Dataset):
    '''
    Fallback if the available validation dataset only returns tensors.
    '''
    def __init__(self, existing_dataset):
        self.dataset = existing_dataset
        self.classes = getattr(existing_dataset, "classes", None)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        if isinstance(image, Image.Image):
            image = raw_transform(image)

        if isinstance(image, torch.Tensor):
            if image.min() < 0 or image.max() > 1:
                image = unnormalize_tensor(image)
            else:
                image = torch.clamp(image.cpu(), 0, 1)

        return image.float(), int(label)


def build_raw_val_dataset():
    '''
    Priority:
    1. Use val_dataset from Member 1/2 cells.
    2. Recreate the same stratified split from DATASET_ROOT/train if available.
    '''
    if "val_dataset" in globals():
        print("Using existing val_dataset from Member 1/2.")
        try:
            return RawSubsetFromImageFolder(val_dataset, raw_transform)
        except Exception as e:
            print("Raw ImageFolder rebuild failed:", e)
            print("Using fallback method.")
            return RawFallbackDataset(val_dataset)

    if "DATASET_ROOT" in globals():
        train_path = Path(DATASET_ROOT) / "train"
    else:
        train_path = CONTENT / "miniimagenet" / "train"

    if not train_path.exists():
        raise RuntimeError(
            "Could not find val_dataset or DATASET_ROOT/train. "
            "Run Member 1/2 dataset cells first."
        )

    print("Rebuilding validation split from:", train_path)
    full_train_raw = datasets.ImageFolder(train_path, transform=raw_transform)

    label_to_indices = defaultdict(list)
    for idx, (_, label) in enumerate(full_train_raw.samples):
        label_to_indices[label].append(idx)

    train_indices, val_indices_rebuilt = [], []
    set_seed(SEED)
    for label, indices in label_to_indices.items():
        random.shuffle(indices)
        cut = max(1, int(0.9 * len(indices)))
        train_indices.extend(indices[:cut])
        val_indices_rebuilt.extend(indices[cut:])

    return Subset(full_train_raw, val_indices_rebuilt)


raw_val_dataset = build_raw_val_dataset()

# Class names
if hasattr(raw_val_dataset, "classes") and raw_val_dataset.classes is not None:
    CLASS_NAMES = list(raw_val_dataset.classes)
elif isinstance(raw_val_dataset, Subset) and hasattr(raw_val_dataset.dataset, "classes"):
    CLASS_NAMES = list(raw_val_dataset.dataset.classes)
elif "CLASS_NAMES" in globals():
    CLASS_NAMES = list(CLASS_NAMES)
else:
    CLASS_NAMES = [str(i) for i in range(64)]

NUM_CLASSES = len(CLASS_NAMES)

print("Validation images:", len(raw_val_dataset))
print("Number of classes:", NUM_CLASSES)
print("First 5 classes:", CLASS_NAMES[:5])

## Cell 30 — Model builders and checkpoint loading

The model heads match Member 1 and Member 2:
- ResNet-18: final `fc` replaced with 64-class layer.
- MobileNetV2: classifier replaced with `Dropout(0.4) + Linear(1280 → 64)`.

In [ ]:
# Cell 30: Model builders and checkpoint loading

def build_resnet18(num_classes):
    '''
    Build ResNet-18 with a new classifier head.
    This matches Member 1.
    '''
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_mobilenetv2(num_classes):
    '''
    Build MobileNetV2 with the same classifier head used by Member 2.
    '''
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes)
    )
    return model


def load_checkpoint_flexible(model, checkpoint_path):
    '''
    Load checkpoints saved as:
    - plain state_dict
    - {"state_dict": ...}
    - {"model_state_dict": ...}
    Handles common prefixes like module. or model.
    '''
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    clean_state = {}
    for key, value in state_dict.items():
        new_key = key
        if new_key.startswith("module."):
            new_key = new_key[len("module."):]
        if new_key.startswith("model."):
            new_key = new_key[len("model."):]
        clean_state[new_key] = value

    missing, unexpected = model.load_state_dict(clean_state, strict=False)

    if missing:
        print("Missing keys preview:", missing[:5])
    if unexpected:
        print("Unexpected keys preview:", unexpected[:5])

    model.to(DEVICE)
    model.eval()
    return model


models_to_evaluate = {}

# ResNet-18
resnet18_model = build_resnet18(NUM_CLASSES)
resnet18_model = load_checkpoint_flexible(resnet18_model, RESNET18_CKPT)
models_to_evaluate["ResNet-18"] = resnet18_model
print("Loaded ResNet-18.")

# MobileNetV2
mobilenetv2_model = build_mobilenetv2(NUM_CLASSES)
mobilenetv2_model = load_checkpoint_flexible(mobilenetv2_model, MOBILENETV2_CKPT)
models_to_evaluate["MobileNetV2"] = mobilenetv2_model
print("Loaded MobileNetV2.")

print("Models ready:", list(models_to_evaluate.keys()))

## Cell 31 — Wavelet compression functions

Compression ratio meaning:

\[
\text{ratio} = \frac{\text{all wavelet coefficients}}{\text{kept coefficients}}
\]

So:
- 2:1 keeps about 50%.
- 5:1 keeps about 20%.
- 10:1 keeps about 10%.

In [ ]:
# Cell 31: Wavelet compression functions

def threshold_coefficients(coeff_array, ratio=2, mode="hard"):
    '''
    Keep the largest coefficients by magnitude.
    Ratio = all coefficients / kept coefficients.
    '''
    flat_abs = np.abs(coeff_array).ravel()
    total = flat_abs.size
    keep = max(1, int(np.ceil(total / ratio)))

    if keep >= total:
        return coeff_array

    threshold = np.partition(flat_abs, total - keep)[total - keep]

    if mode == "hard":
        out = coeff_array.copy()
        out[np.abs(out) < threshold] = 0
        return out

    if mode == "soft":
        return np.sign(coeff_array) * np.maximum(np.abs(coeff_array) - threshold, 0)

    raise ValueError("mode must be 'hard' or 'soft'")


def wavelet_compress_channel(channel, ratio=2, wavelet="haar", level=2, mode="hard"):
    '''
    Apply 2D DWT compression to one image channel.
    '''
    h, w = channel.shape

    coeffs = pywt.wavedec2(channel, wavelet=wavelet, level=level)
    coeff_array, coeff_slices = pywt.coeffs_to_array(coeffs)

    compressed_array = threshold_coefficients(
        coeff_array,
        ratio=ratio,
        mode=mode
    )

    compressed_coeffs = pywt.array_to_coeffs(
        compressed_array,
        coeff_slices,
        output_format="wavedec2"
    )

    reconstructed = pywt.waverec2(compressed_coeffs, wavelet=wavelet)
    reconstructed = reconstructed[:h, :w]

    return np.clip(reconstructed, 0, 1)


def wavelet_compress_image(image_tensor, ratio=2, wavelet="haar", level=2, mode="hard"):
    '''
    Apply wavelet compression to an RGB image tensor.
    Input:  [3,H,W] in [0,1]
    Output: [3,H,W] in [0,1]
    '''
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    output_np = np.zeros_like(image_np)

    for c in range(3):
        output_np[:, :, c] = wavelet_compress_channel(
            image_np[:, :, c],
            ratio=ratio,
            wavelet=wavelet,
            level=level,
            mode=mode
        )

    output_tensor = torch.from_numpy(output_np).permute(2, 0, 1).float()
    return torch.clamp(output_tensor, 0, 1)


class WaveletCompressedDataset(Dataset):
    '''
    Dataset wrapper:
    1. Load raw image.
    2. Apply wavelet compression if ratio is not None.
    3. Normalize with ImageNet mean/std.
    '''
    def __init__(self, base_dataset, ratio=None):
        self.base_dataset = base_dataset
        self.ratio = ratio

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        image = torch.clamp(image, 0, 1)

        if self.ratio is not None:
            image = wavelet_compress_image(
                image,
                ratio=self.ratio,
                wavelet=WAVELET_NAME,
                level=WAVELET_LEVEL,
                mode=THRESHOLD_MODE
            )

        image = normalize_transform(image)
        return image, int(label)


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

## Cell 32 — Original vs compressed images

Use this figure in the slides.

In [ ]:
# Cell 32: Original vs compressed images

def tensor_to_plot_image(x):
    return np.clip(x.permute(1, 2, 0).cpu().numpy(), 0, 1)


def show_original_vs_compressed(dataset, sample_indices=None):
    if sample_indices is None:
        sample_indices = [0, 10, 20, 30]

    titles = ["Original", "2:1", "5:1", "10:1"]

    fig, axes = plt.subplots(
        len(sample_indices),
        len(titles),
        figsize=(12, 3 * len(sample_indices))
    )

    if len(sample_indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(sample_indices):
        image, label = dataset[idx]

        versions = [
            image,
            wavelet_compress_image(image, ratio=2, wavelet=WAVELET_NAME, level=WAVELET_LEVEL, mode=THRESHOLD_MODE),
            wavelet_compress_image(image, ratio=5, wavelet=WAVELET_NAME, level=WAVELET_LEVEL, mode=THRESHOLD_MODE),
            wavelet_compress_image(image, ratio=10, wavelet=WAVELET_NAME, level=WAVELET_LEVEL, mode=THRESHOLD_MODE),
        ]

        for col, img in enumerate(versions):
            axes[row, col].imshow(tensor_to_plot_image(img))
            axes[row, col].axis("off")

            if row == 0:
                axes[row, col].set_title(titles[col])

            if col == 0:
                axes[row, col].set_ylabel(str(CLASS_NAMES[label]), fontsize=8)

    plt.suptitle("Original vs Wavelet Compressed Images")
    plt.tight_layout()

    save_path = M3_FIGURES / "original_vs_compressed.png"
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)


show_original_vs_compressed(raw_val_dataset)

## Cell 33 — Evaluation functions

In [ ]:
# Cell 33: Evaluation functions

criterion = nn.CrossEntropyLoss()


def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.long().to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())

    avg_loss = total_loss / total
    acc = correct / total

    return avg_loss, acc, np.array(y_true), np.array(y_pred)


def build_per_class_accuracy(y_true, y_pred, model_name, compression_label):
    rows = []

    for class_idx in range(NUM_CLASSES):
        mask = (y_true == class_idx)
        support = int(mask.sum())

        if support == 0:
            acc = np.nan
        else:
            acc = float((y_pred[mask] == y_true[mask]).mean())

        rows.append({
            "Model": model_name,
            "Compression": compression_label,
            "Class Index": class_idx,
            "Class Name": CLASS_NAMES[class_idx],
            "Support": support,
            "Class Accuracy": acc,
            "Class Accuracy (%)": 100 * acc if not np.isnan(acc) else np.nan
        })

    return pd.DataFrame(rows)

## Cell 34 — Run the full Member 3 experiment

This evaluates both models on:
- Original validation images.
- 2:1 compressed images.
- 5:1 compressed images.
- 10:1 compressed images.

In [ ]:
# Cell 34: Run the full Member 3 experiment

def run_wavelet_experiment(models_dict):
    summary_rows = []
    per_class_tables = []
    prediction_data = {}

    compression_cases = [("Original", None)] + [(f"{ratio}:1", ratio) for ratio in COMPRESSION_RATIOS]

    for model_name, model in models_dict.items():
        print("\n" + "=" * 70)
        print("Model:", model_name)

        for compression_label, ratio in compression_cases:
            print("-" * 50)
            print("Compression:", compression_label)

            dataset = WaveletCompressedDataset(raw_val_dataset, ratio=ratio)
            loader = make_eval_loader(dataset)

            loss, acc, y_true, y_pred = evaluate_model(model, loader)

            summary_rows.append({
                "Model": model_name,
                "Compression": compression_label,
                "Wavelet": WAVELET_NAME,
                "Level": WAVELET_LEVEL,
                "Threshold Mode": THRESHOLD_MODE,
                "Loss": loss,
                "Accuracy": acc,
                "Accuracy (%)": 100 * acc
            })

            per_class_tables.append(
                build_per_class_accuracy(y_true, y_pred, model_name, compression_label)
            )

            prediction_data[(model_name, compression_label)] = (y_true, y_pred)

            print(f"Loss: {loss:.4f} | Accuracy: {100*acc:.2f}%")

    summary_df = pd.DataFrame(summary_rows)
    per_class_df = pd.concat(per_class_tables, ignore_index=True)

    return summary_df, per_class_df, prediction_data


summary_df, per_class_df, prediction_data = run_wavelet_experiment(models_to_evaluate)

print("\nTable 1. Wavelet Compression Summary")
display(summary_df)

summary_csv = M3_OUTPUT / "wavelet_summary_both_models.csv"
per_class_csv = M3_OUTPUT / "wavelet_per_class_accuracy_both_models.csv"

summary_df.to_csv(summary_csv, index=False)
per_class_df.to_csv(per_class_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", per_class_csv)

## Cell 35 — Baseline sanity check

The original accuracy should be close to:
- ResNet-18: 76.48%
- MobileNetV2: 75.05%

In [ ]:
# Cell 35: Baseline sanity check

reported = {
    "ResNet-18": M1_REPORTED_VAL_ACC,
    "MobileNetV2": M2_REPORTED_VAL_ACC
}

for model_name, expected_acc in reported.items():
    computed = summary_df[
        (summary_df["Model"] == model_name) &
        (summary_df["Compression"] == "Original")
    ]["Accuracy"].iloc[0]

    diff = abs(computed - expected_acc)

    print(f"{model_name}")
    print(f"  Reported baseline : {expected_acc*100:.2f}%")
    print(f"  Recomputed baseline: {computed*100:.2f}%")
    print(f"  Difference        : {diff*100:.2f}%")

    if diff > 0.02:
        print("  Warning: difference is more than 2%. Check dataset split or transforms.")
    else:
        print("  OK: baseline is close.")

## Cell 36 — Accuracy and loss plots

These are the main result plots for your slides.

In [ ]:
# Cell 36: Accuracy and loss plots

def compression_to_order(label):
    if label == "Original":
        return 1
    return int(label.split(":")[0])


plot_df = summary_df.copy()
plot_df["Order"] = plot_df["Compression"].apply(compression_to_order)
plot_df = plot_df.sort_values(["Model", "Order"])

# Accuracy plot
plt.figure(figsize=(8, 5))
for model_name in plot_df["Model"].unique():
    temp = plot_df[plot_df["Model"] == model_name].sort_values("Order")
    plt.plot(temp["Compression"], temp["Accuracy (%)"], marker="o", linewidth=2, label=model_name)

plt.title("Accuracy vs Wavelet Compression Ratio")
plt.xlabel("Compression Ratio")
plt.ylabel("Validation Accuracy (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

acc_plot = M3_FIGURES / "accuracy_vs_compression_both_models.png"
plt.savefig(acc_plot, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", acc_plot)

# Loss plot
plt.figure(figsize=(8, 5))
for model_name in plot_df["Model"].unique():
    temp = plot_df[plot_df["Model"] == model_name].sort_values("Order")
    plt.plot(temp["Compression"], temp["Loss"], marker="o", linewidth=2, label=model_name)

plt.title("Loss vs Wavelet Compression Ratio")
plt.xlabel("Compression Ratio")
plt.ylabel("Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

loss_plot = M3_FIGURES / "loss_vs_compression_both_models.png"
plt.savefig(loss_plot, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", loss_plot)

## Cell 37 — Accuracy drop and robustness comparison

This shows which model is more stable under compression.

In [ ]:
# Cell 37: Accuracy drop and robustness comparison

def build_drop_summary(summary_df):
    rows = []

    for model_name in summary_df["Model"].unique():
        original_acc = summary_df[
            (summary_df["Model"] == model_name) &
            (summary_df["Compression"] == "Original")
        ]["Accuracy"].iloc[0]

        for comp in ["2:1", "5:1", "10:1"]:
            comp_acc = summary_df[
                (summary_df["Model"] == model_name) &
                (summary_df["Compression"] == comp)
            ]["Accuracy"].iloc[0]

            rows.append({
                "Model": model_name,
                "Compression": comp,
                "Original Accuracy (%)": 100 * original_acc,
                "Compressed Accuracy (%)": 100 * comp_acc,
                "Accuracy Drop (%)": 100 * (original_acc - comp_acc)
            })

    return pd.DataFrame(rows)


drop_summary_df = build_drop_summary(summary_df)

print("Table 2. Accuracy Drop Summary")
display(drop_summary_df)

drop_summary_csv = M3_OUTPUT / "accuracy_drop_summary.csv"
drop_summary_df.to_csv(drop_summary_csv, index=False)
print("Saved:", drop_summary_csv)

# Plot accuracy drop
plt.figure(figsize=(8, 5))
for model_name in drop_summary_df["Model"].unique():
    temp = drop_summary_df[drop_summary_df["Model"] == model_name]
    plt.plot(temp["Compression"], temp["Accuracy Drop (%)"], marker="o", linewidth=2, label=model_name)

plt.title("Accuracy Drop vs Compression Ratio")
plt.xlabel("Compression Ratio")
plt.ylabel("Accuracy Drop (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

drop_plot = M3_FIGURES / "accuracy_drop_vs_compression.png"
plt.savefig(drop_plot, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", drop_plot)

## Cell 38 — Class-wise accuracy drop

This answers: which classes are most affected by compression?

In [ ]:
# Cell 38: Class-wise accuracy drop

def compute_class_accuracy_drop(per_class_df):
    rows = []

    for model_name in per_class_df["Model"].unique():
        base = per_class_df[
            (per_class_df["Model"] == model_name) &
            (per_class_df["Compression"] == "Original")
        ][["Class Index", "Class Name", "Class Accuracy"]].rename(
            columns={"Class Accuracy": "Original Class Accuracy"}
        )

        for comp in ["2:1", "5:1", "10:1"]:
            compressed = per_class_df[
                (per_class_df["Model"] == model_name) &
                (per_class_df["Compression"] == comp)
            ][["Class Index", "Class Name", "Class Accuracy"]].rename(
                columns={"Class Accuracy": "Compressed Class Accuracy"}
            )

            merged = base.merge(compressed, on=["Class Index", "Class Name"])
            merged["Model"] = model_name
            merged["Compression"] = comp
            merged["Class Accuracy Drop"] = (
                merged["Original Class Accuracy"] - merged["Compressed Class Accuracy"]
            )
            merged["Class Accuracy Drop (%)"] = 100 * merged["Class Accuracy Drop"]

            rows.append(merged)

    return pd.concat(rows, ignore_index=True)


class_drop_df = compute_class_accuracy_drop(per_class_df)

class_drop_csv = M3_OUTPUT / "class_accuracy_drop_both_models.csv"
class_drop_df.to_csv(class_drop_csv, index=False)

print("Table 3. Largest Class Accuracy Drops")
display(class_drop_df.sort_values("Class Accuracy Drop (%)", ascending=False).head(20))

print("Saved:", class_drop_csv)


def plot_top_class_drops(class_drop_df, compression_label="10:1", top_n=12):
    for model_name in class_drop_df["Model"].unique():
        temp = class_drop_df[
            (class_drop_df["Model"] == model_name) &
            (class_drop_df["Compression"] == compression_label)
        ].sort_values("Class Accuracy Drop (%)", ascending=False).head(top_n)

        plt.figure(figsize=(10, 5))
        plt.bar(temp["Class Name"], temp["Class Accuracy Drop (%)"])
        plt.title(f"Top {top_n} Class Accuracy Drops — {model_name} — {compression_label}")
        plt.xlabel("Class")
        plt.ylabel("Accuracy Drop (%)")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()

        safe_model = model_name.replace("-", "").replace(" ", "_")
        safe_comp = compression_label.replace(":", "to")
        path = M3_FIGURES / f"class_drop_{safe_model}_{safe_comp}.png"
        plt.savefig(path, dpi=220, bbox_inches="tight")
        plt.show()
        print("Saved:", path)


plot_top_class_drops(class_drop_df, "2:1")
plot_top_class_drops(class_drop_df, "5:1")
plot_top_class_drops(class_drop_df, "10:1")

## Cell 39 — Confusion matrices at 10:1

This is optional to show in slides, but useful for GitHub results.

In [ ]:
# Cell 39: Confusion matrices at 10:1

def plot_confusion_for_model(model_name, compression_label="10:1"):
    y_true, y_pred = prediction_data[(model_name, compression_label)]
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))

    fig, ax = plt.subplots(figsize=(12, 12))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[str(i) for i in range(NUM_CLASSES)]
    )
    disp.plot(ax=ax, xticks_rotation=90, values_format="d", colorbar=True)
    plt.title(f"Confusion Matrix — {model_name} — {compression_label} Wavelet Compression")
    plt.tight_layout()

    safe_model = model_name.replace("-", "").replace(" ", "_")
    safe_comp = compression_label.replace(":", "to")
    path = M3_FIGURES / f"confusion_matrix_{safe_model}_{safe_comp}.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", path)


plot_confusion_for_model("ResNet-18", "10:1")
plot_confusion_for_model("MobileNetV2", "10:1")

## Cell 40 — Sample predictions under compression

This gives a visual code demo for the presentation.

In [ ]:
# Cell 40: Sample predictions under compression

def denormalize(x, mean=MEAN, std=STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return torch.clamp(x.cpu() * std + mean, 0, 1)


def show_sample_predictions(model, model_name, ratio=10, n_show=12):
    dataset = WaveletCompressedDataset(raw_val_dataset, ratio=ratio)
    loader = make_eval_loader(dataset)

    images, labels = next(iter(loader))
    images_device = images.to(DEVICE)

    model.eval()
    with torch.no_grad():
        preds = model(images_device).argmax(dim=1).cpu()

    cols = 4
    rows = int(np.ceil(n_show / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = np.array(axes).flatten()

    for i in range(n_show):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        true_name = CLASS_NAMES[int(labels[i])]
        pred_name = CLASS_NAMES[int(preds[i])]
        color = "green" if int(labels[i]) == int(preds[i]) else "red"

        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"T: {true_name}\nP: {pred_name}", color=color, fontsize=8)

    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{model_name} Sample Predictions at {ratio}:1 Compression")
    plt.tight_layout()

    safe_model = model_name.replace("-", "").replace(" ", "_")
    path = M3_FIGURES / f"sample_predictions_{safe_model}_{ratio}to1.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", path)


show_sample_predictions(models_to_evaluate["ResNet-18"], "ResNet-18", ratio=10)
show_sample_predictions(models_to_evaluate["MobileNetV2"], "MobileNetV2", ratio=10)

## Cell 41 — Save final JSON summary

In [ ]:
# Cell 41: Save final JSON summary

final_results = {
    "member": "Member 3",
    "project": "Image Compression and Classification with Deep Learning",
    "task": "Wavelet compression analysis",
    "models": ["ResNet-18", "MobileNetV2"],
    "settings": {
        "image_size": IMG_SIZE,
        "num_classes": NUM_CLASSES,
        "wavelet": WAVELET_NAME,
        "wavelet_level": WAVELET_LEVEL,
        "threshold_mode": THRESHOLD_MODE,
        "compression_ratios": COMPRESSION_RATIOS,
        "normalization_mean": MEAN,
        "normalization_std": STD,
        "seed": SEED
    },
    "reported_baselines": {
        "ResNet-18": M1_REPORTED_VAL_ACC,
        "MobileNetV2": M2_REPORTED_VAL_ACC
    },
    "summary": summary_df.to_dict(orient="records"),
    "accuracy_drop_summary": drop_summary_df.to_dict(orient="records")
}

json_path = M3_OUTPUT / "member3_wavelet_results_summary.json"

with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print("Saved:", json_path)

## Cell 42 — Write README and package Member 3 output

This creates a ZIP file ready for GitHub or Blackboard backup.

In [ ]:
# Cell 42: Write README and package Member 3 output

readme_text = f'''# EE 413 — Member 3: Wavelet Compression Analysis

## Overview
This module covers the required wavelet compression analysis.

Member 3 evaluates how wavelet-based image compression affects image classification performance for:
- ResNet-18 from Member 1
- MobileNetV2 from Member 2

## Compression Setup
| Item | Value |
|---|---|
| Image size | {IMG_SIZE}x{IMG_SIZE} |
| Wavelet | {WAVELET_NAME} |
| Wavelet level | {WAVELET_LEVEL} |
| Thresholding | {THRESHOLD_MODE} |
| Compression ratios | 2:1, 5:1, 10:1 |
| Normalization | ImageNet mean/std |
| Dataset used | Internal validation subset from Mini-ImageNet train split |

## Models Evaluated
| Model | Source |
|---|---|
| ResNet-18 | Member 1 checkpoint: resnet18_best.pth |
| MobileNetV2 | Member 2 checkpoint: mobilenetv2_best.pth |

## Main Outputs
| File | Description |
|---|---|
| wavelet_summary_both_models.csv | Accuracy and loss for both models across compression ratios |
| accuracy_drop_summary.csv | Accuracy drop from original baseline |
| wavelet_per_class_accuracy_both_models.csv | Per-class accuracy for each model and compression ratio |
| class_accuracy_drop_both_models.csv | Class-wise drop caused by compression |
| member3_wavelet_results_summary.json | Final JSON summary |
| figures/original_vs_compressed.png | Visual example of compression |
| figures/accuracy_vs_compression_both_models.png | Main accuracy plot |
| figures/loss_vs_compression_both_models.png | Main loss plot |
| figures/accuracy_drop_vs_compression.png | Robustness comparison |
| figures/class_drop_*.png | Classes most affected by compression |
| figures/confusion_matrix_*.png | Confusion matrices at 10:1 |
| figures/sample_predictions_*.png | Sample predictions under 10:1 compression |

## How to Run
1. Run Member 1 notebook to prepare the dataset and ResNet-18 checkpoint.
2. Run Member 2 notebook to prepare MobileNetV2 checkpoint.
3. Run this notebook from Cell 27 onward.
4. Use the generated CSV files and figures in the presentation.

## Notes
The original validation accuracy is recomputed to verify that the same validation split and transforms are used.
If the recomputed baseline differs by more than 2%, check the dataset split and checkpoint paths.

## Short Conclusion Template
Wavelet compression reduces the number of kept transform coefficients.  
At lower compression such as 2:1, most visual information is preserved.  
At higher compression such as 10:1, some edges, textures, and fine details are removed.  
The accuracy and class-wise drop plots show how this affects ResNet-18 and MobileNetV2 differently.
'''

readme_path = M3_OUTPUT / "README_member3_wavelet.md"
with open(readme_path, "w") as f:
    f.write(readme_text)

print("Saved:", readme_path)

# Package output
zip_path = CONTENT / "member3_wavelet_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in M3_OUTPUT.rglob("*"):
        if file_path.is_file():
            arcname = Path("member3_wavelet_outputs") / file_path.relative_to(M3_OUTPUT)
            zf.write(file_path, arcname=str(arcname))
            print("  +", arcname)

print("\nZip created:", zip_path)
print("Download from Colab file panel, or run:")
print("from google.colab import files; files.download(str(zip_path))")

## Member 3 slide script — short version

Use this for your 5-minute section:

**Start:**  
My part is the wavelet compression analysis. The goal is to test how image compression affects the classification performance of ResNet-18 and MobileNetV2.

**Method:**  
I used the discrete wavelet transform, or DWT, on each image channel. Then I kept only the largest wavelet coefficients based on three compression ratios: 2:1, 5:1, and 10:1. After reconstructing the compressed image, I applied the same ImageNet normalization before sending it to the models.

**Results:**  
First, I checked the original validation accuracy to make sure it matches the baseline. Then I evaluated both models under each compression ratio. The accuracy plot shows how performance changes as compression increases, and the loss plot confirms the same trend.

**Class-wise analysis:**  
I also calculated per-class accuracy drop. This shows which classes were most affected by compression. Classes that depend on fine edges or texture usually lose more accuracy at higher compression.

**Conclusion:**  
The main conclusion is that wavelet compression can reduce image size while keeping useful visual structure, but high compression removes important details. Comparing ResNet-18 and MobileNetV2 shows which model is more robust under compressed inputs.